Limpieza del fichero 01_Ruido_diario_acumulado. Se carga desde Kaggle y se limpia y normalizan los datos

In [1]:
!pip install reverse_geocoder

In [2]:
import pandas as pd
import reverse_geocoder as rg
import kagglehub
from kagglehub import KaggleDatasetAdapter
from sklearn.decomposition import PCA
from scipy.cluster import hierarchy
from scipy.cluster.hierarchy import fcluster
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np



pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [3]:
file_path_01 = "01_Ruido_diario_acumulado.csv"

df_01 = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "raquelahdo/ruido-datasets",
  file_path_01,
    pandas_kwargs={"encoding": "latin-1", "sep": ";"}
)

In [4]:
df_01.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 521356 entries, 0 to 521355
Data columns (total 11 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   NMT     521356 non-null  int64 
 1   Año     521356 non-null  int64 
 2   mes     521356 non-null  int64 
 3   dia     521356 non-null  int64 
 4   tipo    521356 non-null  object
 5   LAeq    521356 non-null  object
 6   L1      521356 non-null  object
 7   L10     521356 non-null  object
 8   L50     521356 non-null  object
 9   L90     521356 non-null  object
 10  L99     521356 non-null  object
dtypes: int64(4), object(7)
memory usage: 43.8+ MB


In [5]:
df_01.describe()

,NMT,Año,mes,dia
count,521356.000000,521356.000000,521356.000000,521356.000000
mean,24.103363,2019.707969,6.405038,15.714055
std,19.153766,3.482799,3.463532,8.818186
min,1.000000,2014.000000,1.000000,1.000000
25%,11.000000,2017.000000,3.000000,8.000000
50%,19.000000,2020.000000,6.000000,16.000000
75%,30.000000,2023.000000,9.000000,23.000000
max,86.000000,2026.000000,12.000000,31.000000


In [6]:
missing_values_01 = df_01.isnull().sum()

display(missing_values_01)

NMT     0
Año     0
mes     0
dia     0
tipo    0
LAeq    0
L1      0
L10     0
L50     0
L90     0
L99     0
dtype: int64

In [6]:
df_01.head()

,NMT,Año,mes,dia,tipo,LAeq,L1,L10,L50,L90,L99
0,3,2014,1,1,D,"57,4","66,6","61,1","54,3","49,1",45
1,3,2014,1,1,E,"58,7","66,6",62,56,"52,1","48,5"
2,3,2014,1,1,N,"65,9","72,5","66,1","60,9","56,2","52,1"
3,3,2014,1,1,T,"62,2","69,3","63,8","56,5","50,3","45,8"
4,4,2014,1,1,D,"67,3","72,4",67,"63,2","59,5","56,2"


Renombrado estandarizado de columnas: Nombres en minusculas, sin espacios, sin acentos, consistentes con Python.
Convertir tipos numéricos correctamente (a float)
Creación de columna Fecha.
Ordenar dataset por fecha.
Limpiar el campo "periodo" D (Diurno), E (Vespertino), N (Nocturno), T (Total)

In [7]:
#creamos una columna fecha para trabajar
#renombramos las columnas con year, month y day. Y resto de columnas
df_01 = df_01.rename(columns={
    "Año": "year",
    "mes": "month",
    "dia": "day",
    "tipo": "periodo",
    "LAeq": "laeq",
    "L1": "l1",
    "L10": "l10",
    "L50": "l50",
    "L90": "l90",
    "L99": "l99"
})

#Se convierten las columnas numéricas a FLOAT (se elimina la coma por punto)
cols_numericas = ["laeq", "l1", "l10", "l50", "l90", "l99"]

for col in cols_numericas:
    df_01[col] = (
        df_01[col]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .astype(float)
    )

#Se construye la columna Fecha (de tipo datetime64). Permite filtrado por año
#Agrupacion semanal/mensal y visualizaciones temporales
df_01["fecha"] = pd.to_datetime(
    df_01[["year", "month", "day"]],
    errors="coerce"
)

#Se ordena el dataset por fecha
df_01 = df_01.sort_values("fecha").reset_index(drop=True)

#Se limpia el campo periodo por valores de etiquetas más claros.
#Se crea columna periodo_desc con la dscripción del periodo
#D:Diurno, E: Vespertino, N: Nosturno, T: Total
df_01["periodo"] = df_01["periodo"].str.upper()
map_periodos = {
    "D": "Diurno",
    "E": "Vespertino",
    "N": "Nocturno",
    "T": "Total"
}
df_01["periodo_desc"] = df_01["periodo"].map(map_periodos)

#Se crean columnas derivadas útiles para gráficos
#Columna: mes en formato nombre
df_01["mes_nombre"] = df_01["fecha"].dt.month_name(locale="es_ES.utf8")

#Columna: Año-mes 
df_01["year_month"] = df_01["fecha"].dt.to_period("M").astype(str)

#Columna: semana del año
df_01["week"] = df_01["fecha"].dt.isocalendar().week

#Se eliminan filas con problemas en fecha y laeq (NaN). Deja dataset limpio y usable
df_01 = df_01.dropna(subset=["fecha", "laeq"])


In [8]:
df_01.head()

,NMT,year,month,day,periodo,laeq,l1,l10,l50,l90,l99,fecha,periodo_desc,mes_nombre,year_month,week
0,3,2014,1,1,D,57.4,66.6,61.1,54.3,49.1,45.0,2014-01-01,Diurno,Enero,2014-01,1
1,30,2014,1,1,D,55.6,64.7,59.2,51.9,45.9,43.5,2014-01-01,Diurno,Enero,2014-01,1
2,29,2014,1,1,T,57.2,67.4,62.1,49.0,36.6,29.8,2014-01-01,Total,Enero,2014-01,1
3,29,2014,1,1,N,55.0,66.7,57.4,41.5,30.9,29.2,2014-01-01,Nocturno,Enero,2014-01,1
4,29,2014,1,1,E,57.4,67.4,62.5,50.2,41.7,36.7,2014-01-01,Vespertino,Enero,2014-01,1


In [9]:
df_01.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 521356 entries, 0 to 521355
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   NMT           521356 non-null  int64         
 1   year          521356 non-null  int64         
 2   month         521356 non-null  int64         
 3   day           521356 non-null  int64         
 4   periodo       521356 non-null  object        
 5   laeq          521356 non-null  float64       
 6   l1            521356 non-null  float64       
 7   l10           521356 non-null  float64       
 8   l50           521356 non-null  float64       
 9   l90           521356 non-null  float64       
 10  l99           521356 non-null  float64       
 11  fecha         521356 non-null  datetime64[ns]
 12  periodo_desc  521356 non-null  object        
 13  mes_nombre    521356 non-null  object        
 14  year_month    521356 non-null  object        
 15  week          521

NECESIDAD DE HABILITAR LA GESTIÓN MASIVA DE DATOS
The number of rows in your dataset is greater than the maximum allowed (5000).

Try enabling the VegaFusion data transformer which raises this limit by pre-evaluating data
transformations in Python.
    >> import altair as alt
    >> alt.data_transformers.enable("vegafusion")

In [10]:
#Se activa VegaFusion
import altair as alt
alt.data_transformers.enable("vegafusion")


DataTransformerRegistry.enable('vegafusion')

In [11]:
pip install "vegafusion[embed]>=1.5.0"

Note: you may need to restart the kernel to use updated packages.


In [12]:
 pip install "vl-convert-python>=1.6.0"

Note: you may need to restart the kernel to use updated packages.


In [13]:

import altair as alt

# Desactivar motores que intentan usar vegafusion
alt.renderers.set_embed_options(actions=False)
alt.data_transformers.enable


<bound method PluginRegistry.enable of DataTransformerRegistry(active='vegafusion', registered=['csv', 'default', 'json', 'vegafusion'])>

In [14]:
#Tendencia temporal de LAeq (línea)

df_daily = df_01.groupby(['fecha', 'periodo'], as_index=False)['laeq'].mean()


chart_tendencia = alt.Chart(df_daily).mark_line().encode(
    x=alt.X('fecha:T', title='Fecha'),
    y=alt.Y('laeq:Q', title='LAeq'),
    color=alt.Color('periodo:N', title='Periodo'),
    tooltip=['fecha:T', 'laeq:Q', 'periodo:N']
).properties(
    title='Evolución temporal del nivel acústico (LAeq) en Madrid',
    width=800,
    height=300
)

chart_tendencia

alt.Chart(...)